# Fly / Wirehead — Interactive 3D App via Cloudflare Tunnel

Runs the original local server (`flywirehead run`) inside the Kaggle kernel and exposes it through a **Cloudflare Quick Tunnel** (`cloudflared`) to view the 3D scene, live dopamine telemetry, and keyboard controls directly in your browser.

**Key steps in this notebook:**
1. Install dependencies, clone repository, and configure the C++ compiler.
2. Download the MaleCNS connectome (~1.1 GB) and video shorts.
3. Download the `cloudflared` binary.
4. Patch origin/session local-only validation locks and launch both the tunnel & simulation server.
5. Monitor live logs and shut down when finished.

## 1. Setup: packages, source, data directory, compiler

In [1]:
import importlib, subprocess, sys, os, shutil, pathlib

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs], check=True)

pip_install("pyarrow>=16.0.0", "tqdm", "yt-dlp")

REPO_URL = "https://github.com/mattyhempstead/fly-wirehead.git"
REPO = pathlib.Path("/kaggle/working/fly-wirehead")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))

DATA_DIR = pathlib.Path("/kaggle/working/flywirehead-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ["FLYWIREHEAD_DATA"] = str(DATA_DIR)

def ensure_cxx_compiler():
    if shutil.which("c++"):
        return shutil.which("c++")
    if not shutil.which("g++"):
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", "g++"], check=False)
    gpp = shutil.which("g++")
    if not gpp:
        raise RuntimeError("No C++ compiler available — turn on internet and re-run.")
    bindir = pathlib.Path("/kaggle/working/bin")
    bindir.mkdir(parents=True, exist_ok=True)
    link = bindir / "c++"
    if not link.exists():
        link.symlink_to(gpp)
    os.environ["PATH"] = f"{bindir}:{os.environ['PATH']}"
    return str(link)

print("c++ resolves to:", ensure_cxx_compiler())
print("Repo:", REPO, " Data dir:", DATA_DIR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Cloning into '/kaggle/working/fly-wirehead'...


c++ resolves to: /usr/bin/c++
Repo: /kaggle/working/fly-wirehead  Data dir: /kaggle/working/flywirehead-data


## 2. Prepare the connectome and video feed

In [2]:
from flywirehead.data import prepare as prepare_connectome
prepare_connectome()
print("Connectome ready.")

{"dataset": "malecns_v1", "neurons": 166700, "edges": 25582938, "synaptic_contacts": 124177617, "retina_total": 3377, "retina_mapped": 3335, "retina_unmapped": 42, "projection_confidence_median": 1.0, "projection_below_80_percent": 15, "uncertain_sign_neurons": 3718, "retina_model": "R1-R6 luminance-only. Column inferred from all contacts onto annotated L1/L2/L3; modal column. Experimental overlapping viewport projection, not calibrated retinal angles.", "visual_dynamics": "Photoreceptors and lamina are graded in vivo. This experiment uses an explicit LIF proxy, low-pass luminance drive and tonic lamina current; it is not validated fly vision.", "motor_interface": "Stonkfly uses DNp20 mean right-minus-left firing with a DNpe017 spike gate for buy/sell/hold. This is an engineered trading interface.", "training": "The compiled graph is the baseline. Runtime adds the documented candidate KC-to-MBON07/11 plasticity and R8-to-aMe12 sign assumption."}
{"release": "MaleCNS v1.0", "neurons": 1

# if the download_real_videos_ doesnt work from first try run the cell again until it does and make sure internet is enabled !

In [4]:
def download_real_videos():
    for tool in ("yt-dlp", "ffmpeg", "ffprobe"):
        if not shutil.which(tool):
            raise RuntimeError(f"{tool} not found on PATH")
    subprocess.run([sys.executable, str(REPO / "scripts" / "download_videos.py")], cwd=str(REPO), check=True)

try:
    download_real_videos()
    playlist = (REPO / "dist" / "media" / "playlist.json")
    print("Videos ready:", playlist.exists())
except Exception as e:
    print("Video download failed — the server will run, but the phone screen will be blank.")
    print("Reason:", repr(e))

[youtube] Extracting URL: https://www.youtube.com/shorts/HOe8Ur6H8x4
[youtube] HOe8Ur6H8x4: Downloading webpage


[youtube] HOe8Ur6H8x4: Downloading visionos player API JSON
[youtube] HOe8Ur6H8x4: Downloading m3u8 information
[info] HOe8Ur6H8x4: Downloading 1 format(s): 136+140
[info] Writing video metadata as JSON to: /kaggle/working/fly-wirehead/runs/video-downloads/HOe8Ur6H8x4.info.json
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/HOe8Ur6H8x4.f136.mp4
[download] Download completed
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/HOe8Ur6H8x4.f140.m4a
[download] Download completed
[Merger] Merging formats into "/kaggle/working/fly-wirehead/runs/video-downloads/HOe8Ur6H8x4.mp4"
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/HOe8Ur6H8x4.f136.mp4 (pass -k to keep)
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/HOe8Ur6H8x4.f140.m4a (pass -k to keep)
01 · 5.2s · Housefly Sound - Flies ❤️ (1)
[youtube] Extracting URL: https://www.youtube.com/shorts/rUmhjdFVPFo
[youtube] rUmhjdFVPFo: Downloading web

[youtube] rUmhjdFVPFo: Downloading visionos player API JSON
[youtube] rUmhjdFVPFo: Downloading m3u8 information
[info] rUmhjdFVPFo: Downloading 1 format(s): 136+140
[info] Writing video metadata as JSON to: /kaggle/working/fly-wirehead/runs/video-downloads/rUmhjdFVPFo.info.json
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/rUmhjdFVPFo.f136.mp4
[download] Download completed
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/rUmhjdFVPFo.f140.m4a
[download] Download completed
[Merger] Merging formats into "/kaggle/working/fly-wirehead/runs/video-downloads/rUmhjdFVPFo.mp4"
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/rUmhjdFVPFo.f136.mp4 (pass -k to keep)
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/rUmhjdFVPFo.f140.m4a (pass -k to keep)
02 · 7.0s · This is the reason Flies are not wanted to land on food. 🔥👀😎🫵
[youtube] Extracting URL: https://www.youtube.com/shorts/lYrSza4cYaE
[youtu

[youtube] lYrSza4cYaE: Downloading visionos player API JSON
[youtube] lYrSza4cYaE: Downloading m3u8 information
[info] lYrSza4cYaE: Downloading 1 format(s): 136+140
[info] Writing video metadata as JSON to: /kaggle/working/fly-wirehead/runs/video-downloads/lYrSza4cYaE.info.json
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/lYrSza4cYaE.f136.mp4
[download] Download completed
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/lYrSza4cYaE.f140.m4a
[download] Download completed
[Merger] Merging formats into "/kaggle/working/fly-wirehead/runs/video-downloads/lYrSza4cYaE.mp4"
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/lYrSza4cYaE.f140.m4a (pass -k to keep)
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/lYrSza4cYaE.f136.mp4 (pass -k to keep)
03 · 5.0s · Housefly Sound
[youtube] Extracting URL: https://www.youtube.com/shorts/db5JqXQekmE
[youtube] db5JqXQekmE: Downloading webpage


[youtube] db5JqXQekmE: Downloading visionos player API JSON
[youtube] db5JqXQekmE: Downloading m3u8 information
[info] db5JqXQekmE: Downloading 1 format(s): 137+140
[info] Writing video metadata as JSON to: /kaggle/working/fly-wirehead/runs/video-downloads/db5JqXQekmE.info.json
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/db5JqXQekmE.f137.mp4
[download] Download completed
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/db5JqXQekmE.f140.m4a
[download] Download completed
[Merger] Merging formats into "/kaggle/working/fly-wirehead/runs/video-downloads/db5JqXQekmE.mp4"
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/db5JqXQekmE.f140.m4a (pass -k to keep)
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/db5JqXQekmE.f137.mp4 (pass -k to keep)
04 · 14.3s · Amazing close-up of a fly! #shorts
[youtube] Extracting URL: https://www.youtube.com/shorts/PBWmPoLjVvA
[youtube] PBWmPoLjVvA: Downloadi

[youtube] PBWmPoLjVvA: Downloading visionos player API JSON
[youtube] PBWmPoLjVvA: Downloading m3u8 information
[info] PBWmPoLjVvA: Downloading 1 format(s): 298+140-20
[info] Writing video metadata as JSON to: /kaggle/working/fly-wirehead/runs/video-downloads/PBWmPoLjVvA.info.json
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/PBWmPoLjVvA.f298.mp4
[download] Download completed
[download] Destination: /kaggle/working/fly-wirehead/runs/video-downloads/PBWmPoLjVvA.f140-20.m4a
[download] Download completed
[Merger] Merging formats into "/kaggle/working/fly-wirehead/runs/video-downloads/PBWmPoLjVvA.mp4"
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/PBWmPoLjVvA.f140-20.m4a (pass -k to keep)
Deleting original file /kaggle/working/fly-wirehead/runs/video-downloads/PBWmPoLjVvA.f298.mp4 (pass -k to keep)
05 · 4.0s · Unveiling the Short but Dynamic Life of Houseflies: Exploring Their Average Two-Week Lifespan| SGK E
5 portrait Shorts ready, 15

In [14]:
import pathlib

repo = pathlib.Path("/kaggle/working/fly-wirehead")
html_files = list(repo.glob("dist/**/*.html")) + list(repo.glob("*.html"))

hud_snippet = """
<!-- INJECTED FLY-WIREHEAD CONTROL HUD -->
<div id="fly-controls-hud" style="
    position: fixed;
    bottom: 24px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 999999;
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    background: rgba(18, 18, 24, 0.88);
    backdrop-filter: blur(12px);
    -webkit-backdrop-filter: blur(12px);
    padding: 10px 14px;
    border-radius: 14px;
    border: 1px solid rgba(255, 255, 255, 0.18);
    box-shadow: 0 10px 30px rgba(0, 0, 0, 0.6);
    user-select: none;
">
    <style>
        .fly-btn {
            background: rgba(255, 255, 255, 0.08);
            color: #f1f5f9;
            border: 1px solid rgba(255, 255, 255, 0.15);
            padding: 8px 14px;
            border-radius: 8px;
            font-family: system-ui, -apple-system, sans-serif;
            font-size: 13px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.15s ease;
            display: inline-flex;
            align-items: center;
            gap: 6px;
        }
        .fly-btn:hover {
            background: rgba(255, 255, 255, 0.2);
            border-color: rgba(255, 255, 255, 0.35);
            transform: translateY(-1px);
        }
        .fly-btn:active {
            transform: translateY(1px) scale(0.98);
        }
        .fly-btn-dopamine {
            background: #e11d48 !important;
            border-color: #f43f5e !important;
            color: #ffffff !important;
        }
        .fly-btn-dopamine:hover {
            background: #f43f5e !important;
            box-shadow: 0 0 14px rgba(225, 29, 72, 0.6);
        }
    </style>

    <button class="fly-btn" onclick="sendFlyKey('ArrowDown', 'ArrowDown', 40)">⏭️ Next Short</button>
    <button class="fly-btn" onclick="sendFlyKey(' ', 'Space', 32)">⏯️ Pause / Play</button>
    <button class="fly-btn fly-btn-dopamine" onclick="sendFlyKey('p', 'KeyP', 80)">⚡ Dopamine (P)</button>
    <button class="fly-btn" onclick="sendFlyKey('c', 'KeyC', 67)">🎥 Camera (C)</button>
    <button class="fly-btn" onclick="toggleFlyFullscreen()">⛶ Fullscreen</button>
    <button class="fly-btn" onclick="sendFlyKey('m', 'KeyM', 77)">🔊 Audio (M)</button>
    <button class="fly-btn" onclick="sendFlyKey('s', 'KeyS', 83)">💾 Save (S)</button>
</div>

<script>
function sendFlyKey(key, code, keyCode) {
    ['keydown', 'keyup'].forEach(type => {
        const ev = new KeyboardEvent(type, {
            key: key,
            code: code,
            keyCode: keyCode,
            which: keyCode,
            bubbles: true,
            cancelable: true,
            view: window
        });
        window.dispatchEvent(ev);
        document.dispatchEvent(ev);
        if (document.body) document.body.dispatchEvent(ev);
    });
}

function toggleFlyFullscreen() {
    if (!document.fullscreenElement) {
        document.documentElement.requestFullscreen().catch(() => {});
    } else {
        document.exitFullscreen().catch(() => {});
    }
    sendFlyKey('f', 'KeyF', 70);
}
</script>
<!-- END FLY-WIREHEAD CONTROL HUD -->
"""

patched_count = 0
for file_path in html_files:
    content = file_path.read_text(encoding="utf-8")
    if "id=\"fly-controls-hud\"" not in content and "</body>" in content:
        new_content = content.replace("</body>", f"{hud_snippet}\n</body>")
        file_path.write_text(new_content, encoding="utf-8")
        patched_count += 1
        print(f"Patched: {file_path}")

print(f"Done! {patched_count} HTML file(s) updated with on-screen UI controls.")

Patched: /kaggle/working/fly-wirehead/dist/index.html
Done! 1 HTML file(s) updated with on-screen UI controls.


## 3. Download `cloudflared`

In [5]:
CLOUDFLARED = pathlib.Path("/kaggle/working/cloudflared")
if not CLOUDFLARED.exists():
    subprocess.run([
        "curl", "-sL", "-o", str(CLOUDFLARED),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ], check=True)
    CLOUDFLARED.chmod(0o755)

subprocess.run([str(CLOUDFLARED), "--version"], check=True)

cloudflared version 2026.9.0 (built 2026-09-09-16:30 UTC)


CompletedProcess(args=['/kaggle/working/cloudflared', '--version'], returncode=0)

## 4. Patch security checks & launch server + tunnel

This cell automatically cleans any previous processes, applies an AST transformation to remove local-only origin and WebSocket session locks, establishes the Cloudflare quick tunnel, and starts the simulation server. Click the link it generates to watch !

In [64]:
# ============================================================
# FLYWIREHEAD KAGGLE LAUNCHER — PINGGY (NO CLOUDFLARE)
# ============================================================

import os
import sys
import time
import re
import atexit
import pathlib
import subprocess
import shutil
import urllib.request
import urllib.error
import ast

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

PORT = 4173

LOG_DIR = pathlib.Path("/kaggle/working/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

repo_path = pathlib.Path(
    globals().get("REPO", "/kaggle/working/fly-wirehead")
)

data_dir = pathlib.Path(
    globals().get("DATA_DIR", "/kaggle/working/flywirehead-data")
)

LOCAL_URL = f"http://127.0.0.1:{PORT}"

# ------------------------------------------------------------
# PROCESS HELPERS
# ------------------------------------------------------------

def stop_process(proc, name="process"):
    if proc is None:
        return

    try:
        if proc.poll() is None:
            print(f"Stopping {name}...")
            proc.terminate()

            try:
                proc.wait(timeout=3)
            except subprocess.TimeoutExpired:
                print(f"Force killing {name}...")
                proc.kill()
                proc.wait(timeout=2)
    except Exception as e:
        print(f"Cleanup warning for {name}: {e}")


# Stop previous instances
stop_process(globals().get("server_proc"), "old Flywirehead server")
stop_process(globals().get("tunnel_proc"), "old tunnel")

# Free port
try:
    subprocess.run(
        ["fuser", "-k", f"{PORT}/tcp"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        timeout=5,
        check=False
    )
except Exception:
    pass

time.sleep(1)

# ------------------------------------------------------------
# VERIFY REPOSITORY & CLEAN CACHE
# ------------------------------------------------------------

if not repo_path.exists():
    raise RuntimeError(
        f"Flywirehead repository does not exist:\n{repo_path}\n"
        "Run the setup cell first."
    )

data_dir.mkdir(parents=True, exist_ok=True)

try:
    subprocess.run(["git", "checkout", "--", "."], cwd=str(repo_path), capture_output=True)
except Exception:
    pass

for base in [repo_path]:
    if base.exists():
        for p in base.rglob("__pycache__"):
            shutil.rmtree(p, ignore_errors=True)

print("=" * 70)
print("FLYWIREHEAD LAUNCH (PINGGY)")
print("=" * 70)
print(f"Repository : {repo_path}")
print(f"Data       : {data_dir}")
print(f"Local URL  : {LOCAL_URL}")
print("=" * 70)

# ------------------------------------------------------------
# DISABLE AUTH & SECURITY CHECKS VIA AST REWRITING
# ------------------------------------------------------------

search_dirs = [repo_path]
try:
    import flywirehead
    pkg_dir = pathlib.Path(flywirehead.__file__).parent
    search_dirs.append(pkg_dir)
    for p in list(pkg_dir.rglob("__pycache__")):
        shutil.rmtree(p, ignore_errors=True)
except Exception:
    pass

all_py_files = set()
for d in search_dirs:
    all_py_files.update(d.rglob("*.py"))

class FullSecurityBypasser(ast.NodeTransformer):
    def __init__(self):
        self.modified = False

    def visit_If(self, node):
        target_phrases = [
            "local origin", "origin required", "origin not allowed",
            "local session", "session required", "valid local session",
            "invalid session", "unauthorized origin", "session expired"
        ]
        has_security_err = False
        for child in ast.walk(node):
            if isinstance(child, ast.Constant) and isinstance(child.value, str):
                if any(phrase in child.value.lower() for phrase in target_phrases):
                    has_security_err = True
                    break
        if has_security_err:
            node.test = ast.Constant(value=False)
            self.modified = True
        self.generic_visit(node)
        return node

    def visit_FunctionDef(self, node):
        fn_name = node.name.lower()
        if any(fn_name.startswith(p) for p in ["is_", "check_", "validate_", "_check_", "_is_", "_validate_"]):
            if any(t in fn_name for t in ["origin", "local", "session", "auth"]):
                node.body.insert(0, ast.Return(value=ast.Constant(value=True)))
                self.modified = True
        self.generic_visit(node)
        return node

for file_path in all_py_files:
    try:
        source = file_path.read_text(encoding="utf-8")
        lowered = source.lower()
        if any(term in lowered for term in ["local origin", "session required", "valid local session", "check_origin", "check_session"]):
            tree = ast.parse(source)
            transformer = FullSecurityBypasser()
            new_tree = transformer.visit(tree)
            if transformer.modified:
                file_path.write_text(ast.unparse(new_tree), encoding="utf-8")
                print(f"Patched security checks in: {file_path.name}")
    except Exception as e:
        print(f"Notice on {file_path.name}: {e}")

# ------------------------------------------------------------
# HTML CONTROL DOCK
# ------------------------------------------------------------

safe_hud = r"""
<!-- ==========================================================
     FLYWIREHEAD CONTROL DOCK
     ========================================================== -->

<div id="fly-controls-dock">

<style>

#fly-controls-dock {
    position: fixed;
    bottom: 18px;
    left: 50%;
    transform: translateX(-50%);

    z-index: 2147483647;

    display: flex;
    flex-wrap: wrap;
    justify-content: center;

    gap: 8px;

    background: rgba(15, 23, 42, 0.94);

    backdrop-filter: blur(12px);
    -webkit-backdrop-filter: blur(12px);

    padding: 9px 12px;

    border-radius: 13px;

    border: 1px solid rgba(255,255,255,0.20);

    box-shadow:
        0 8px 24px rgba(0,0,0,0.55);

    user-select: none;
    -webkit-user-select: none;

    font-family:
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif;
}

#fly-controls-dock .dock-btn {

    appearance: none;
    -webkit-appearance: none;

    background: rgba(255,255,255,0.12);

    color: white;

    border:
        1px solid rgba(255,255,255,0.25);

    padding: 8px 12px;

    border-radius: 8px;

    font-family: inherit;

    font-size: 13px;
    font-weight: 600;

    cursor: pointer;

    outline: none;

    user-select: none;
    -webkit-user-select: none;

    touch-action: manipulation;
}

#fly-controls-dock .dock-btn:hover {
    background: rgba(255,255,255,0.23);
}

#fly-controls-dock .dock-btn:active {
    transform: scale(0.95);
}

#fly-controls-dock .dock-btn-red {
    background: #e11d48 !important;
    border-color: #f43f5e !important;
}

</style>


<button
    type="button"
    class="dock-btn"
    onmousedown="flyDockSwipe(event, this)"
    ontouchstart="flyDockSwipe(event, this)"
>
    ⏭️ Force swipe to the next Short
</button>


<button
    type="button"
    class="dock-btn"
    onmousedown="flyDockKey(event, ' ', 'Space')"
    ontouchstart="flyDockKey(event, ' ', 'Space')"
>
    ⏯️ Pause / Play
</button>


<button
    type="button"
    class="dock-btn dock-btn-red"
    onmousedown="flyDockKey(event, 'p', 'KeyP')"
    ontouchstart="flyDockKey(event, 'p', 'KeyP')"
>
    ⚡ Dopamine (P)
</button>


<button
    type="button"
    class="dock-btn"
    onmousedown="flyDockKey(event, 'c', 'KeyC')"
    ontouchstart="flyDockKey(event, 'c', 'KeyC')"
>
    🎥 Camera (C)
</button>


<div style="display: flex; align-items: center; color: rgba(255, 255, 255, 0.7); font-size: 13px; font-weight: 500; padding: 0 6px;">
    🔊 Press M to turn on audio
</div>


<button
    type="button"
    class="dock-btn"
    onmousedown="flyDockKey(event, 's', 'KeyS')"
    ontouchstart="flyDockKey(event, 's', 'KeyS')"
>
    💾 Save (S)
</button>


<button
    type="button"
    class="dock-btn"
    onmousedown="flyDockFullscreen(event)"
    ontouchstart="flyDockFullscreen(event)"
>
    ⛶ Fullscreen
</button>


<script>

(function () {

    window.flyDockKey = function (event, key, code) {

        if (event) {
            event.preventDefault();
            event.stopPropagation();
        }

        let target = document.activeElement;

        if (
            !target ||
            target === document.body ||
            target === document.documentElement ||
            target.closest &&
            target.closest('#fly-controls-dock')
        ) {
            target = document.body;
        }

        const down = new KeyboardEvent(
            'keydown',
            {
                key: key,
                code: code,
                bubbles: true,
                cancelable: true,
                composed: true,
                ctrlKey: false,
                altKey: false,
                shiftKey: false,
                metaKey: false,
                repeat: false
            }
        );

        const up = new KeyboardEvent(
            'keyup',
            {
                key: key,
                code: code,
                bubbles: true,
                cancelable: true,
                composed: true,
                ctrlKey: false,
                altKey: false,
                shiftKey: false,
                metaKey: false,
                repeat: false
            }
        );

        try {
            target.dispatchEvent(down);
        } catch (_) {}

        try {
            document.dispatchEvent(down);
        } catch (_) {}

        try {
            window.dispatchEvent(down);
        } catch (_) {}

        setTimeout(function () {

            try {
                target.dispatchEvent(up);
            } catch (_) {}

            try {
                document.dispatchEvent(up);
            } catch (_) {}

            try {
                window.dispatchEvent(up);
            } catch (_) {}

        }, 35);

        setTimeout(function () {

            try {

                const app =
                    document.querySelector(
                        'canvas, video, main, #root, #app'
                    );

                if (app && typeof app.focus === 'function') {
                    app.focus({preventScroll: true});
                }

            } catch (_) {}

        }, 50);
    };

    window.flyDockFullscreen = function (event) {

        if (event) {
            event.preventDefault();
            event.stopPropagation();
        }

        try {

            if (!document.fullscreenElement) {

                document.documentElement
                    .requestFullscreen()
                    .catch(function () {});

            } else {

                document
                    .exitFullscreen()
                    .catch(function () {});

            }

        } catch (_) {}
    };

    // --- NEW: SWIPE LOGIC WITH VISUAL FEEDBACK ---
    window.flyDockSwipe = function(event, btn) {
        window.flyDockKey(event, 'ArrowDown', 'ArrowDown');
        
        setTimeout(function() {
            try {
                const wheel = new WheelEvent('wheel', { deltaY: 800, bubbles: true, cancelable: true, view: window });
                window.dispatchEvent(wheel);
                document.dispatchEvent(wheel);
                const canvas = document.querySelector('canvas');
                if (canvas) canvas.dispatchEvent(wheel);
            } catch(e) {}
        }, 10);
        
        const oldText = btn.innerHTML;
        btn.innerHTML = "✅ Swiped!";
        setTimeout(function() { btn.innerHTML = oldText; }, 800);
    };

    // --- NEW: 2-SECOND REFRESH WATCHDOG TO KEEP KEYS ALIVE ---
    setInterval(function () {
        try {
            const app = document.querySelector('canvas, video, main, #root, #app');
            if (app && document.activeElement !== app) {
                if (app.tagName && app.tagName.toLowerCase() === 'canvas') {
                    app.setAttribute('tabindex', '0');
                }
                app.focus({preventScroll: true});
            }
        } catch (_) {}
    }, 2000);

})();

</script>

</div>
<!-- ==========================================================
     END FLYWIREHEAD CONTROL DOCK
     ========================================================== -->
"""

# ------------------------------------------------------------
# INJECT UI
# ------------------------------------------------------------

html_targets = []
dist_dir = repo_path / "dist"

if dist_dir.exists():
    html_targets.extend(dist_dir.rglob("*.html"))

html_targets.extend(repo_path.glob("*.html"))

# Remove duplicates
html_targets = list(dict.fromkeys(html_targets))

print()
print(f"Found {len(html_targets)} HTML file(s).")

for h in html_targets:
    try:
        c = h.read_text(encoding="utf-8")

        # Remove previous copy if present
        if "<!-- ==========================================================" in c and \
           "FLYWIREHEAD CONTROL DOCK" in c:

            start_marker = "<!-- ==========================================================\n     FLYWIREHEAD CONTROL DOCK"
            end_marker = "<!-- ==========================================================\n     END FLYWIREHEAD CONTROL DOCK"

            start = c.find(start_marker)
            if start != -1:
                end = c.find("</div>", start)
                if end != -1:
                    end += len("</div>")
                    c = c[:start] + c[end:]

        if "</body>" in c.lower():
            # Preserve normal document structure
            idx = c.lower().rfind("</body>")

            c = (
                c[:idx]
                + "\n"
                + safe_hud
                + "\n"
                + c[idx:]
            )

            h.write_text(c, encoding="utf-8")
            print(f"Injected controls: {h}")

    except Exception as e:
        print(f"Could not modify {h}: {e}")

# ------------------------------------------------------------
# START FLYWIREHEAD FIRST
# ------------------------------------------------------------

env = os.environ.copy()

# These are legitimate origin configuration values.
# The tunnel hostname is not known yet, so don't pretend it is.
env.pop("FLYWIREHEAD_EXTRA_HOST", None)
env.pop("FLYWIREHEAD_ALLOWED_ORIGIN", None)

server_log_path = LOG_DIR / "flywirehead.log"

server_log = open(
    server_log_path,
    "w",
    buffering=1,
)

print()
print("Starting Flywirehead...")

server_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "flywirehead",
        "--data",
        str(data_dir),
        "run",
        "--no-browser",
        "--port",
        str(PORT),
    ],
    cwd=str(repo_path),
    env=env,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    text=True,
)

atexit.register(
    stop_process,
    server_proc,
    "Flywirehead"
)

# ------------------------------------------------------------
# WAIT FOR LOCAL SERVER
# ------------------------------------------------------------

def check_http(url, timeout=2):
    try:
        req = urllib.request.Request(
            url,
            headers={
                "User-Agent": "Kaggle-Flywirehead-Launcher"
            },
        )
        with urllib.request.urlopen(req, timeout=timeout) as response:
            return response.status
    except Exception:
        return None

print()
print("Waiting for Flywirehead to become ready...")

local_ready = False
deadline = time.monotonic() + 45

while time.monotonic() < deadline:
    if server_proc.poll() is not None:
        print()
        print("=" * 70)
        print("FLYWIREHEAD CRASHED")
        print("=" * 70)
        try:
            print(server_log_path.read_text(
                encoding="utf-8",
                errors="replace"
            ))
        except Exception:
            pass
        raise RuntimeError("Flywirehead exited before becoming ready.")

    status = check_http(LOCAL_URL)
    if status is not None:
        print(f"Local server READY: {LOCAL_URL} (HTTP {status})")
        local_ready = True
        break
    time.sleep(0.5)

if not local_ready:
    print()
    print("--- Flywirehead log ---")
    try:
        print(server_log_path.read_text(encoding="utf-8", errors="replace"))
    except Exception:
        pass
    raise RuntimeError(f"Flywirehead did not become reachable on {LOCAL_URL}")

# ------------------------------------------------------------
# START PINGGY TUNNEL
# ------------------------------------------------------------

print()
print("Starting Pinggy SSH Tunnel...")

tunnel_log_path = LOG_DIR / "tunnel.log"
tunnel_log_path.write_text("")

tunnel_log = open(tunnel_log_path, "w", buffering=1)

tunnel_proc = subprocess.Popen(
    [
        "ssh",
        "-p", "443",
        "-R0:localhost:4173",
        "-o", "StrictHostKeyChecking=no",
        "-o", "ServerAliveInterval=30",
        "qr@a.pinggy.io"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    text=True,
)

atexit.register(stop_process, tunnel_proc, "Pinggy tunnel")

# ------------------------------------------------------------
# EXTRACT & VERIFY PUBLIC URL
# ------------------------------------------------------------

public_url = None
deadline = time.monotonic() + 45

while time.monotonic() < deadline:
    if tunnel_proc.poll() is not None:
        print()
        print("=" * 70)
        print("PINGGY TUNNEL EXITED")
        print("=" * 70)
        print(tunnel_log_path.read_text(encoding="utf-8", errors="replace"))
        raise RuntimeError("SSH tunnel exited before creating a URL.")

    try:
        text = tunnel_log_path.read_text(encoding="utf-8", errors="replace")
        matches = re.findall(
            r"https://[a-zA-Z0-9.-]+(?:pinggy\.net|pinggy-free\.link|pinggy\.link)",
            text
        )
        valid_urls = [u for u in matches if "dashboard.pinggy.io" not in u]
        if valid_urls:
            public_url = valid_urls[0]
            break
    except Exception:
        pass
    time.sleep(0.5)

if not public_url:
    print()
    print("=" * 70)
    print("NO PUBLIC URL")
    print("=" * 70)
    print(tunnel_log_path.read_text(encoding="utf-8", errors="replace"))
    raise RuntimeError("Pinggy did not provide a public URL.")

print()
print(f"Pinggy URL obtained: {public_url}")

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print()
print("=" * 70)
print("🚀 FLYWIREHEAD IS READY")
print("=" * 70)
print()
print("LOCAL:")
print(f"  {LOCAL_URL}")
print()
print("PUBLIC:")
print(f"  {public_url}")
print()
print("KEYBOARD:")
print("  Real keyboard input is preserved.")
print()
print("SCREEN CONTROLS:")
print("  Next Short       → ArrowDown")
print("  Pause / Play     → Space")
print("  Dopamine         → P")
print("  Camera           → C")
print("  Audio            → M")
print("  Save             → S")
print("  Fullscreen       → browser fullscreen")
print()
print("=" * 70)
print()
print("Keep this Kaggle cell running while using the link.")
print("=" * 70)

# ------------------------------------------------------------
# STREAM LOG
# ------------------------------------------------------------

try:
    with open(server_log_path, "r", encoding="utf-8", errors="replace") as f:
        f.seek(0, os.SEEK_END)
        while True:
            line = f.readline()
            if line:
                print("[Flywirehead]", line.rstrip())
            else:
                if server_proc.poll() is not None:
                    print()
                    print("Flywirehead stopped.")
                    break
                if tunnel_proc.poll() is not None:
                    print()
                    print("Pinggy tunnel stopped.")
                    break
                time.sleep(0.5)

except KeyboardInterrupt:
    print()
    print("Launcher interrupted.")

finally:
    print("Cleaning up...")
    stop_process(tunnel_proc, "Pinggy tunnel")
    stop_process(server_proc, "Flywirehead")

FLYWIREHEAD LAUNCH (PINGGY)
Repository : /kaggle/working/fly-wirehead
Data       : /kaggle/working/flywirehead-data
Local URL  : http://127.0.0.1:4173
Patched security checks in: server.py

Found 1 HTML file(s).
Injected controls: /kaggle/working/fly-wirehead/dist/index.html

Starting Flywirehead...

Waiting for Flywirehead to become ready...
Local server READY: http://127.0.0.1:4173 (HTTP 200)

Starting Pinggy SSH Tunnel...

Pinggy URL obtained: https://avimd-35-254-160-24.free.pinggy.net

🚀 FLYWIREHEAD IS READY

LOCAL:
  http://127.0.0.1:4173

PUBLIC:
  https://avimd-35-254-160-24.free.pinggy.net

KEYBOARD:
  Real keyboard input is preserved.

SCREEN CONTROLS:
  Next Short       → ArrowDown
  Pause / Play     → Space
  Dopamine         → P
  Camera           → C
  Audio            → M
  Save             → S
  Fullscreen       → browser fullscreen


Keep this Kaggle cell running while using the link.
[Flywirehead] Brain ready: 166,700 neurons, 25,582,938 connections; 20400.0 ms simula

## 5. Check status / see logs

In [10]:
def status(proc, name):
    alive = proc is not None and proc.poll() is None
    print(f"{name}: {'running' if alive else 'stopped'} (pid {proc.pid if proc else '-'})")

status(tunnel_proc, "cloudflared")
status(server_proc, "flywirehead server")
print()
print("--- last 20 lines of flywirehead.log ---")
if (LOG_DIR / "flywirehead.log").exists():
    print("\n".join((LOG_DIR / "flywirehead.log").read_text().splitlines()[-20:]))

cloudflared: stopped (pid 834)
flywirehead server: running (pid 844)

--- last 20 lines of flywirehead.log ---
Fly Wirehead: http://127.0.0.1:4173
Ctrl-C saves the brain and exits.
Brain ready: 166,700 neurons, 25,582,938 connections; 3750.0 ms simulated
Saving brain state…


## 6. Shut it down when you're done

In [52]:
stop(server_proc, "server_proc")
stop(tunnel_proc, "tunnel_proc")

## (Optional) Prefer to keep using ngrok instead of Cloudflare?

```python
# pip install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url
tunnel_host = public_url.removeprefix("https://")
os.environ["FLYWIREHEAD_EXTRA_HOST"] = tunnel_host
```